In [1]:
# 01_data_exploration.ipynb
# Country Music Lyrics NLP Project - Initial Setup & Exploration

import json
import pandas as pd
from pathlib import Path
import spacy

print("✅ Environment ready for Country Music Lyrics NLP project")

# Load sample lyrics
data_file = Path("../data/raw/lyrics_sample.json")
with open(data_file, encoding="utf-8") as f:
    lyrics_data = json.load(f)

df = pd.DataFrame(lyrics_data)
print(f"\n✅ Loaded {len(df)} sample songs")
print("\nSample data preview:")
print(df[['song', 'artist', 'year', 'theme']])

# Load spaCy model
nlp = spacy.load("en_core_web_sm")
print("\n✅ spaCy 'en_core_web_sm' model loaded successfully")

# Test NER on first song
doc = nlp(df['lyrics'].iloc[0])
print("\n✅ NER Entities from 'Jolene' lyrics:")
for ent in doc.ents:
    print(f"  {ent.text} → {ent.label_}")

✅ Environment ready for Country Music Lyrics NLP project

✅ Loaded 3 sample songs

Sample data preview:
                      song            artist  year      theme
0                   Jolene      Dolly Parton  1973   jealousy
1         Before He Cheats  Carrie Underwood  2005    revenge
2  The House That Built Me   Miranda Lambert  2009  nostalgia

✅ spaCy 'en_core_web_sm' model loaded successfully

✅ NER Entities from 'Jolene' lyrics:
  Jolene → PERSON
  Jolene → GPE
  Jolene → PERSON
  Jolene → PERSON
  Jolene → PERSON
  Jolene → PERSON
  Jolene → PERSON
  Jolene → PERSON


In [2]:
# Basic Information Extraction from Lyrics

def extract_entities(text):
    doc = nlp(text)
    entities = [(ent.text, ent.label_) for ent in doc.ents]
    return entities

# Apply to all songs
df['entities'] = df['lyrics'].apply(extract_entities)

print("✅ Entity Extraction Complete")
print("\nExtracted Entities per Song:")
for i, row in df.iterrows():
    print(f"\n{row['song']} by {row['artist']}:")
    for ent in row['entities']:
        print(f"   • {ent[0]} → {ent[1]}")

✅ Entity Extraction Complete

Extracted Entities per Song:

Jolene by Dolly Parton:
   • Jolene → PERSON
   • Jolene → GPE
   • Jolene → PERSON
   • Jolene → PERSON
   • Jolene → PERSON
   • Jolene → PERSON
   • Jolene → PERSON
   • Jolene → PERSON

Before He Cheats by Carrie Underwood:

The House That Built Me by Miranda Lambert:


In [3]:
# Improved Entity Extraction with Custom Patterns

from spacy.matcher import Matcher

# Create custom matcher for better results on lyrics
matcher = Matcher(nlp.vocab)

# Pattern for repeated names (like "Jolene, Jolene")
pattern_name = [{"LOWER": {"IN": ["jolene", "carrie", "miranda"]}}]
matcher.add("REPEATED_NAME", [pattern_name])

# Extract improved entities
def extract_improved_entities(text, song_title, artist):
    doc = nlp(text)
    entities = [(ent.text, ent.label_) for ent in doc.ents]
    
    # Add custom matches
    matches = matcher(doc)
    for match_id, start, end in matches:
        span = doc[start:end]
        entities.append((span.text, "SONG_REF"))
    
    return entities

# Apply to dataframe
df['improved_entities'] = df.apply(
    lambda row: extract_improved_entities(row['lyrics'], row['song'], row['artist']), 
    axis=1
)

print("✅ Improved Entity Extraction Complete")
print("\nResults:")
for i, row in df.iterrows():
    print(f"\n{row['song']} by {row['artist']}:")
    for ent in row['improved_entities']:
        print(f"   • {ent[0]} → {ent[1]}")

✅ Improved Entity Extraction Complete

Results:

Jolene by Dolly Parton:
   • Jolene → PERSON
   • Jolene → GPE
   • Jolene → PERSON
   • Jolene → PERSON
   • Jolene → PERSON
   • Jolene → PERSON
   • Jolene → PERSON
   • Jolene → PERSON
   • Jolene → SONG_REF
   • Jolene → SONG_REF
   • Jolene → SONG_REF
   • Jolene → SONG_REF
   • Jolene → SONG_REF
   • Jolene → SONG_REF
   • Jolene → SONG_REF
   • Jolene → SONG_REF

Before He Cheats by Carrie Underwood:

The House That Built Me by Miranda Lambert:


In [4]:
print("Current DataFrame columns:")
print(df.columns.tolist())

print("\nFull improved entities output:")
for i, row in df.iterrows():
    print(f"\n=== {row['song']} by {row['artist']} ===")
    print("Lyrics snippet:", row['lyrics'][:150] + "...")
    print("Extracted:", row['improved_entities'])

Current DataFrame columns:
['song', 'artist', 'year', 'lyrics', 'theme', 'entities', 'improved_entities']

Full improved entities output:

=== Jolene by Dolly Parton ===
Lyrics snippet: Jolene, Jolene, Jolene, Jolene
I'm begging of you please don't take my man
Jolene, Jolene, Jolene, Jolene
Please don't take him just because you can

...
Extracted: [('Jolene', 'PERSON'), ('Jolene', 'GPE'), ('Jolene', 'PERSON'), ('Jolene', 'PERSON'), ('Jolene', 'PERSON'), ('Jolene', 'PERSON'), ('Jolene', 'PERSON'), ('Jolene', 'PERSON'), ('Jolene', 'SONG_REF'), ('Jolene', 'SONG_REF'), ('Jolene', 'SONG_REF'), ('Jolene', 'SONG_REF'), ('Jolene', 'SONG_REF'), ('Jolene', 'SONG_REF'), ('Jolene', 'SONG_REF'), ('Jolene', 'SONG_REF')]

=== Before He Cheats by Carrie Underwood ===
Lyrics snippet: Right now he's probably slow dancing with a bleached-blonde tramp
And she's probably getting frisky
Right now he's probably buying her some fruity lit...
Extracted: []

=== The House That Built Me by Miranda Lambert ===
L

In [5]:
# Stronger Rule-based + Custom Entity Extraction for Lyrics

from spacy.matcher import Matcher
import re

nlp = spacy.load("en_core_web_sm")
matcher = Matcher(nlp.vocab)

# Better patterns for lyrics
patterns = [
    # Repeated names / titles
    [{"LOWER": {"IN": ["jolene", "carrie", "miranda", "cheats"]}}],
    # Common emotional / relational terms
    [{"LOWER": {"IN": ["man", "house", "tramp", "beauty", "eyes", "heart"]}}]
]
matcher.add("LYRICS_ENTITY", patterns)

def extract_lyrics_entities(text, song, artist):
    doc = nlp(text.lower())
    entities = []
    
    # spaCy default
    for ent in doc.ents:
        entities.append((ent.text.title(), ent.label_))
    
    # Custom matcher
    matches = matcher(doc)
    for match_id, start, end in matches:
        span = doc[start:end]
        entities.append((span.text.title(), "LYRICS_REF"))
    
    # Simple regex for common patterns
    if re.search(r'jolene', text, re.I):
        entities.append(("Jolene", "SONG_TITLE"))
    if "house" in text.lower():
        entities.append(("House", "THEME"))
    if "cheats" in text.lower() or "tramp" in text.lower():
        entities.append(("Infidelity", "THEME"))
    
    # Remove duplicates while preserving order
    seen = set()
    unique = []
    for e in entities:
        key = (e[0].lower(), e[1])
        if key not in seen:
            seen.add(key)
            unique.append(e)
    
    return unique

# Apply to all songs
df['strong_entities'] = df.apply(
    lambda row: extract_lyrics_entities(row['lyrics'], row['song'], row['artist']), 
    axis=1
)

print("✅ Stronger Entity Extraction Complete\n")
for i, row in df.iterrows():
    print(f"{row['song']} by {row['artist']}:")
    for ent in row['strong_entities']:
        print(f"   • {ent[0]:<20} → {ent[1]}")
    print("-" * 50)

✅ Stronger Entity Extraction Complete

Jolene by Dolly Parton:
   • Jolene               → LYRICS_REF
   • Man                  → LYRICS_REF
   • Beauty               → LYRICS_REF
   • Eyes                 → LYRICS_REF
   • Jolene               → SONG_TITLE
--------------------------------------------------
Before He Cheats by Carrie Underwood:
   • Tramp                → LYRICS_REF
   • Infidelity           → THEME
--------------------------------------------------
The House That Built Me by Miranda Lambert:
--------------------------------------------------


In [6]:
# Enhanced Lyrics Information Extraction Pipeline

import re

def clean_lyrics(text):
    text = re.sub(r'\n+', ' ', text)  # normalize newlines
    return text.strip()

def extract_lyrics_info(row):
    text = clean_lyrics(row['lyrics'].lower())
    song = row['song']
    artist = row['artist']
    entities = []
    
    # 1. Song Title & Artist
    entities.append((song, "SONG_TITLE"))
    entities.append((artist, "ARTIST"))
    
    # 2. Common Themes (rule-based)
    themes = []
    if any(word in text for word in ['jolene', 'begging', 'man']):
        themes.append("Jealousy")
    if any(word in text for word in ['cheats', 'tramp', 'frisky', 'dancing']):
        themes.append("Infidelity/Revenge")
    if any(word in text for word in ['house', 'built', 'healing', 'place']):
        themes.append("Nostalgia/Home")
    if themes:
        for t in themes:
            entities.append((t, "THEME"))
    
    # 3. Key Entities & Phrases with regex + spaCy
    doc = nlp(text)
    for ent in doc.ents:
        if ent.label_ in ["PERSON", "GPE", "ORG"]:
            entities.append((ent.text.title(), ent.label_))
    
    # 4. Simple keyword entities
    keywords = ["man", "beauty", "eyes", "heart", "tramp", "whiskey"]
    for kw in keywords:
        if kw in text:
            entities.append((kw.title(), "KEY_ELEMENT"))
    
    # Deduplicate
    seen = set()
    unique = []
    for e in entities:
        key = (e[0].lower(), e[1])
        if key not in seen:
            seen.add(key)
            unique.append(e)
    
    return unique

# Apply pipeline
df['extracted_info'] = df.apply(extract_lyrics_info, axis=1)

print("✅ Enhanced Information Extraction Pipeline Complete\n")
for i, row in df.iterrows():
    print(f"{row['song']} by {row['artist']}")
    for ent in row['extracted_info']:
        print(f"   • {ent[0]:<25} → {ent[1]}")
    print("-" * 60)

✅ Enhanced Information Extraction Pipeline Complete

Jolene by Dolly Parton
   • Jolene                    → SONG_TITLE
   • Dolly Parton              → ARTIST
   • Jealousy                  → THEME
   • Man                       → KEY_ELEMENT
   • Beauty                    → KEY_ELEMENT
   • Eyes                      → KEY_ELEMENT
------------------------------------------------------------
Before He Cheats by Carrie Underwood
   • Before He Cheats          → SONG_TITLE
   • Carrie Underwood          → ARTIST
   • Infidelity/Revenge        → THEME
   • Tramp                     → KEY_ELEMENT
   • Whiskey                   → KEY_ELEMENT
------------------------------------------------------------
The House That Built Me by Miranda Lambert
   • The House That Built Me   → SONG_TITLE
   • Miranda Lambert           → ARTIST
   • Nostalgia/Home            → THEME
------------------------------------------------------------


In [9]:
# Save processed data - Fixed path
from pathlib import Path

# Use absolute path to avoid relative path issues
project_root = Path.cwd().parent  # Go up from notebooks/
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

output_file = processed_dir / "lyrics_extracted.json"

df.to_json(output_file, orient="records", indent=2)
print(f"✅ Successfully saved processed data to:\n{output_file}")
print(f"File size: {output_file.stat().st_size} bytes")

✅ Successfully saved processed data to:
c:\Users\matts\PythonProjects\country-music-lyrics-nlp\country-music-lyrics-nlp\data\processed\lyrics_extracted.json
File size: 3841 bytes


In [10]:
# Load additional lyrics and combine datasets
more_file = Path("../data/raw/more_lyrics.json")
with open(more_file, encoding="utf-8") as f:
    more_data = json.load(f)

df_more = pd.DataFrame(more_data)

# Combine both datasets
df_combined = pd.concat([df, df_more], ignore_index=True)
print(f"✅ Combined dataset now has {len(df_combined)} songs")

# Re-run extraction on combined data
df_combined['extracted_info'] = df_combined.apply(extract_lyrics_info, axis=1)

print("\nCombined Results Summary:")
for i, row in df_combined.iterrows():
    print(f"• {row['song']} by {row['artist']} → {len(row['extracted_info'])} entities")

✅ Combined dataset now has 6 songs

Combined Results Summary:
• Jolene by Dolly Parton → 6 entities
• Before He Cheats by Carrie Underwood → 5 entities
• The House That Built Me by Miranda Lambert → 3 entities
• Wagon Wheel by Darius Rucker → 2 entities
• Tennessee Whiskey by Chris Stapleton → 4 entities
• Hurricane by Luke Combs → 2 entities
